> **Public reproducibility copy.** Notebook outputs and execution counts were cleared before public packaging to avoid local-path/account leakage. The scientific code and configuration cells are preserved. See the repository README for required data paths and reproducibility scope.

# Exploratory STARE Rerun 594 — v8 Self-Contained

This version fixes:

```text
ModuleNotFoundError: No module named 'production_handcrafted_extractor'
```

The exact recovered handcrafted extractor is embedded directly inside the notebook. No external `.py` module, path edit, or `sys.path` modification is required.


## Required folders only

```text
AMD_DATASET_594_PRODUCTION_v1/
AMD_BASELINE_REPRODUCTION_v1/
└── feature_cache/
STARE_data/
├── NORMAL/
├── AMD_DRY/
└── AMD_WET/
```

The separate `production_handcrafted_extractor.py` file is no longer required.


In [ ]:
# Install only when necessary, then restart the kernel.
# %pip install -q numpy pandas scipy scikit-image scikit-learn imbalanced-learn opencv-python mahotas matplotlib timm torch torchvision pillow


In [ ]:
from __future__ import annotations

"""
Exploratory STARE rerun using the exact recovered production source.

Primary scientific configuration
--------------------------------
- Training data: AMD_DATASET_594_PRODUCTION_v1 only.
- Image view for production equivalence: ROI_594 / roi_release_path.
- Handcrafted vector: Color9 -> LBP18 -> Haralick5.
- ViT model: timm "vit_base_patch16_224", num_classes=0.
- ViT normalization: mean/std = 0.5 for all channels.
- Fusion order: Handcrafted32 -> ViT768 = 800D.
- Pipeline: SMOTE -> StandardScaler -> PCA-128 -> RBF-SVM.
- SVM: C=10, gamma="scale", class_weight="balanced".
- Prespecified primary seed: 42.
- Secondary robustness seeds: 42, 123, 2026, 3407, 7777.

STARE is not used for fitting, tuning, dimensionality selection, threshold
selection, feature selection, or checkpoint selection.
"""

from pathlib import Path
import hashlib
import json
import os
import pickle
import platform
import random
import time
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy.stats import binomtest
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE

import torch
import timm
import torchvision.transforms as T

# ============================================================
# EXACT PRODUCTION HANDCRAFTED EXTRACTOR — INLINED
# ============================================================
# Recovered from the original experiment and production-reproduction notebooks.
# This block is intentionally embedded so the notebook does not depend on
# `production_handcrafted_extractor.py` or the current working directory.

import cv2
import mahotas
from scipy.stats import skew
from skimage.feature import local_binary_pattern

LBP_P = 16
LBP_R = 2

FEATURE_NAMES = [
    "R_mean", "R_std", "R_skew",
    "G_mean", "G_std", "G_skew",
    "B_mean", "B_std", "B_skew",
    *[f"LBP_bin_{i}" for i in range(18)],
    "Haralick_d1_ASM",
    "Haralick_d1_Contrast",
    "Haralick_d1_Correlation",
    "Haralick_d1_SSVariance",
    "Haralick_d1_IDM",
]


def extract_handcrafted_32d(image_path: str | Path) -> np.ndarray:
    """
    Exact 32D extractor recovered from:
    - Pap_II_new_FIXED_SVM_PCA_Sensitivity notebook; and
    - AMD_Baseline_Reproduction_Production_v1 notebook.

    Feature order:
    1. RGB color moments: 9D
    2. Uniform LBP P=16, R=2: 18D
    3. First five mean Mahotas Haralick features, distance=1: 5D
    """
    image_path = str(image_path)
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise ValueError(f"OpenCV could not read: {image_path}")

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    features: list[float] = []

    # 9 RGB color moments.
    for channel_index in range(3):
        channel = img_rgb[:, :, channel_index]
        features.extend([
            float(np.mean(channel)),
            float(np.std(channel)),
            float(skew(channel.flatten())),
        ])

    # 18-bin uniform LBP.
    lbp = local_binary_pattern(
        img_gray,
        LBP_P,
        LBP_R,
        method="uniform",
    )
    n_bins = LBP_P + 2
    hist_lbp, _ = np.histogram(
        lbp.ravel(),
        bins=n_bins,
        range=(0, n_bins),
        density=True,
    )
    features.extend(hist_lbp.astype(float).tolist())

    # First five directional-mean Mahotas Haralick features at distance=1.
    haralick_d1 = mahotas.features.haralick(
        img_gray,
        distance=1,
    ).mean(axis=0)
    features.extend(haralick_d1[:5].astype(float).tolist())

    array = np.asarray(features, dtype=np.float32)
    if array.shape != (32,):
        raise ValueError(
            f"Expected 32 handcrafted features for {image_path}; "
            f"found {array.shape}"
        )

    return np.nan_to_num(
        array,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )



CONFIG = {
    "DATASET_ROOT": Path(r"AMD_DATASET_594_PRODUCTION_v1"),
    "FEATURE_CACHE_ROOT": Path(r"AMD_BASELINE_REPRODUCTION_v1/feature_cache"),
    "STARE_ROOT": Path(r"STARE_data"),
    "OUTPUT_ROOT": Path(r"AMD_EXPLORATORY_STARE_RERUN_594_v8_SELF_CONTAINED"),

    "PRIMARY_SEED": 42,
    "ROBUSTNESS_SEEDS": [42, 123, 2026, 3407, 7777],

    "VIT_MODEL": "vit_base_patch16_224",
    "VIT_INPUT_SIZE": 224,
    "VIT_BATCH_SIZE": 16,

    "SVM_C": 10.0,
    "SVM_GAMMA": "scale",
    "CLASS_WEIGHT": "balanced",
    "SMOTE_K_NEIGHBORS": 5,
    "PCA_COMPONENTS": 128,

    "VERIFY_ALL_594_HANDCRAFTED_FEATURES": True,
    "EQUIVALENCE_ATOL": 1e-5,
    "EQUIVALENCE_RTOL": 1e-5,

    "BOOTSTRAP_RESAMPLES": 5000,
    "BOOTSTRAP_SEED": 20260728,

    "FAIL_ON_SINGLE_CLASS_PREDICTION": True,
    "RESUME": True,
}


METHODS = [
    "FROZEN_VIT_SVM",
    "PROPOSED_HYBRID_PCA128_SVM",
]
EXPECTED_CLASSES = ["AMD_DRY", "AMD_WET", "NORMAL"]
SUPPORTED_EXTENSIONS = {
    ".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".ppm"
}


def seed_all(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def save_csv(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def save_json(payload: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    temporary.replace(path)


def majority_vote(series: pd.Series):
    return series.value_counts().index[0]


def compute_metrics(y_true, y_pred, labels):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=labels,
            average="macro",
            zero_division=0,
        ),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    }


def bootstrap_paired_difference(
    y_true,
    frozen_pred,
    hybrid_pred,
    n_resamples,
    seed,
):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    frozen_pred = np.asarray(frozen_pred)
    hybrid_pred = np.asarray(hybrid_pred)
    labels = sorted(np.unique(y_true).tolist())
    n = len(y_true)
    values = np.empty((n_resamples, 3), dtype=float)

    for index in range(n_resamples):
        sampled = rng.integers(0, n, size=n)
        frozen_metrics = np.asarray([
            accuracy_score(y_true[sampled], frozen_pred[sampled]),
            f1_score(
                y_true[sampled],
                frozen_pred[sampled],
                labels=labels,
                average="macro",
                zero_division=0,
            ),
            balanced_accuracy_score(
                y_true[sampled], frozen_pred[sampled]
            ),
        ])
        hybrid_metrics = np.asarray([
            accuracy_score(y_true[sampled], hybrid_pred[sampled]),
            f1_score(
                y_true[sampled],
                hybrid_pred[sampled],
                labels=labels,
                average="macro",
                zero_division=0,
            ),
            balanced_accuracy_score(
                y_true[sampled], hybrid_pred[sampled]
            ),
        ])
        values[index] = hybrid_metrics - frozen_metrics

    output = {"bootstrap_resamples": int(n_resamples)}
    for metric_index, metric_name in enumerate(
        ["accuracy", "macro_f1", "balanced_accuracy"]
    ):
        vector = values[:, metric_index]
        output[f"{metric_name}_difference_mean"] = float(vector.mean())
        output[f"{metric_name}_ci95_low"] = float(
            np.quantile(vector, 0.025)
        )
        output[f"{metric_name}_ci95_high"] = float(
            np.quantile(vector, 0.975)
        )
    return output


def load_production_manifest() -> pd.DataFrame:
    root = CONFIG["DATASET_ROOT"]
    manifest_path = (
        root / "provenance" / "release_manifest_594_sha256.csv"
    )
    folds_path = root / "outer_fold_assignments.csv"

    if not manifest_path.exists():
        raise FileNotFoundError(manifest_path)
    if not folds_path.exists():
        raise FileNotFoundError(folds_path)

    manifest = pd.read_csv(manifest_path, dtype={"sample_id": str})
    folds = pd.read_csv(folds_path, dtype={"sample_id": str})

    required_manifest = {
        "sample_id",
        "class_label",
        "source_dataset",
        "roi_release_path",
    }
    missing = sorted(required_manifest - set(manifest.columns))
    if missing:
        raise ValueError(f"Manifest columns missing: {missing}")

    manifest["sample_id"] = manifest["sample_id"].str.strip()
    folds["sample_id"] = folds["sample_id"].str.strip()

    frame = manifest.merge(
        folds[["sample_id", "outer_fold"]],
        on="sample_id",
        how="left",
        validate="one_to_one",
        suffixes=("", "_fold"),
    )
    if "outer_fold_fold" in frame.columns:
        if "outer_fold" in frame.columns:
            frame = frame.drop(columns=["outer_fold"])
        frame = frame.rename(
            columns={"outer_fold_fold": "outer_fold"}
        )

    if len(frame) != 594:
        raise ValueError(f"Expected 594 rows; found {len(frame)}")
    if frame["sample_id"].duplicated().any():
        raise ValueError("Duplicate production sample IDs")
    if frame["outer_fold"].isna().any():
        raise ValueError("Missing immutable outer-fold assignments")

    frame["image_path"] = frame["roi_release_path"].map(
        lambda value: str(root / str(value))
    )
    missing_paths = [
        path
        for path in frame["image_path"]
        if not Path(path).is_file()
    ]
    if missing_paths:
        raise FileNotFoundError(
            f"{len(missing_paths)} ROI paths are missing. "
            f"First: {missing_paths[:3]}"
        )

    frame = frame.sort_values("sample_id").reset_index(drop=True)
    observed_counts = frame["class_label"].value_counts().to_dict()
    expected_counts = {
        "NORMAL": 287,
        "AMD_DRY": 139,
        "AMD_WET": 168,
    }
    if observed_counts != expected_counts:
        raise ValueError(
            f"Unexpected class counts: {observed_counts}"
        )
    return frame


def load_training_features(frame: pd.DataFrame):
    root = CONFIG["FEATURE_CACHE_ROOT"]
    paths = {
        "hc": root / "handcrafted_32d.npy",
        "hc_ids": root / "handcrafted_sample_ids.csv",
        "vit": root / "vit_b16_768d.npy",
        "vit_ids": root / "vit_b16_sample_ids.csv",
    }
    for path in paths.values():
        if not path.exists():
            raise FileNotFoundError(path)

    handcrafted = np.load(paths["hc"]).astype(np.float32)
    vit = np.load(paths["vit"]).astype(np.float32)
    handcrafted_ids = pd.read_csv(
        paths["hc_ids"], dtype={"sample_id": str}
    )["sample_id"].str.strip().tolist()
    vit_ids = pd.read_csv(
        paths["vit_ids"], dtype={"sample_id": str}
    )["sample_id"].str.strip().tolist()
    manifest_ids = frame["sample_id"].tolist()

    if handcrafted.shape != (594, 32):
        raise ValueError(handcrafted.shape)
    if vit.shape != (594, 768):
        raise ValueError(vit.shape)
    if handcrafted_ids != manifest_ids:
        raise ValueError("Handcrafted cache IDs do not match manifest")
    if vit_ids != manifest_ids:
        raise ValueError("ViT cache IDs do not match manifest")
    return handcrafted, vit


def verify_exact_handcrafted_cache(
    frame: pd.DataFrame,
    canonical: np.ndarray,
    output_root: Path,
) -> dict:
    if not CONFIG["VERIFY_ALL_594_HANDCRAFTED_FEATURES"]:
        indices = np.linspace(
            0, len(frame) - 1, num=60, dtype=int
        )
    else:
        indices = np.arange(len(frame))

    rows = []
    observed_vectors = []
    for completed, index in enumerate(indices, start=1):
        row = frame.iloc[int(index)]
        vector = extract_handcrafted_32d(row["image_path"])
        observed_vectors.append(vector)

        difference = vector - canonical[int(index)]
        rows.append({
            "sample_id": row["sample_id"],
            "image_path": row["image_path"],
            "mean_absolute_difference": float(
                np.mean(np.abs(difference))
            ),
            "maximum_absolute_difference": float(
                np.max(np.abs(difference))
            ),
            "allclose": bool(
                np.allclose(
                    vector,
                    canonical[int(index)],
                    atol=CONFIG["EQUIVALENCE_ATOL"],
                    rtol=CONFIG["EQUIVALENCE_RTOL"],
                )
            ),
        })
        if completed % 50 == 0 or completed == len(indices):
            print(
                f"Handcrafted equivalence: "
                f"{completed}/{len(indices)}"
            )

    audit = pd.DataFrame(rows)
    save_csv(
        audit,
        output_root / "handcrafted_feature_equivalence_by_sample.csv",
    )
    observed = np.vstack(observed_vectors)
    expected = canonical[indices]
    difference = observed - expected

    result = {
        "n_verified": int(len(indices)),
        "feature_order": [
            "RGB_color_moments_9D",
            "uniform_LBP_18D",
            "Mahotas_Haralick_first5_5D",
        ],
        "mean_absolute_difference": float(
            np.mean(np.abs(difference))
        ),
        "maximum_absolute_difference": float(
            np.max(np.abs(difference))
        ),
        "allclose": bool(
            np.allclose(
                observed,
                expected,
                atol=CONFIG["EQUIVALENCE_ATOL"],
                rtol=CONFIG["EQUIVALENCE_RTOL"],
            )
        ),
        "failed_samples": int((~audit["allclose"]).sum()),
    }
    save_json(
        result,
        output_root / "HANDCRAFTED_EQUIVALENCE_GATE.json",
    )
    if not result["allclose"]:
        worst = audit.sort_values(
            "maximum_absolute_difference",
            ascending=False,
        ).head(10)
        raise RuntimeError(
            "Exact-source handcrafted equivalence failed. "
            "Do not evaluate STARE.\n"
            + worst.to_string(index=False)
        )
    return result


def build_stare_manifest() -> pd.DataFrame:
    root = CONFIG["STARE_ROOT"]
    if not root.exists():
        raise FileNotFoundError(root)

    rows = []
    for label in ["NORMAL", "AMD_DRY", "AMD_WET"]:
        folder = root / label
        if not folder.exists():
            raise FileNotFoundError(folder)
        images = sorted(
            path
            for path in folder.rglob("*")
            if path.is_file()
            and path.suffix.lower() in SUPPORTED_EXTENSIONS
        )
        if not images:
            raise ValueError(f"No images in {folder}")

        for ordinal, image_path in enumerate(images, start=1):
            relative = image_path.relative_to(root).as_posix()
            token = hashlib.sha256(
                relative.encode("utf-8")
            ).hexdigest()[:12]
            rows.append({
                "sample_id": (
                    f"STARE_{label}_{ordinal:04d}_{token}"
                ),
                "image_path": relative,
                "harmonized_label": label,
                "original_folder_label": label,
                "label_rule": (
                    "Predefined folder label used as a "
                    "rule-harmonized three-class category"
                ),
                "file_sha256": sha256_file(image_path),
                "file_size_bytes": image_path.stat().st_size,
            })

    frame = pd.DataFrame(rows).sort_values(
        ["harmonized_label", "image_path"]
    ).reset_index(drop=True)

    if frame["sample_id"].duplicated().any():
        raise ValueError("Duplicate STARE sample IDs")
    if frame["file_sha256"].duplicated().any():
        raise ValueError("Exact duplicate STARE images detected")

    frame["resolved_image_path"] = frame["image_path"].map(
        lambda value: str(root / str(value))
    )
    return frame.sort_values("sample_id").reset_index(drop=True)


def extract_vit_features(frame: pd.DataFrame):
    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )
    model = timm.create_model(
        CONFIG["VIT_MODEL"],
        pretrained=True,
        num_classes=0,
    ).to(device).eval()

    transform = T.Compose([
        T.Resize(
            (CONFIG["VIT_INPUT_SIZE"], CONFIG["VIT_INPUT_SIZE"])
        ),
        T.ToTensor(),
        T.Normalize(
            mean=(0.5, 0.5, 0.5),
            std=(0.5, 0.5, 0.5),
        ),
    ])

    feature_parts = []
    sample_ids = []
    start = time.perf_counter()

    with torch.no_grad():
        for batch_start in range(
            0, len(frame), CONFIG["VIT_BATCH_SIZE"]
        ):
            batch_frame = frame.iloc[
                batch_start:
                batch_start + CONFIG["VIT_BATCH_SIZE"]
            ]
            tensors = []
            batch_ids = []

            for _, row in batch_frame.iterrows():
                with Image.open(
                    row["resolved_image_path"]
                ) as image:
                    tensors.append(
                        transform(image.convert("RGB"))
                    )
                batch_ids.append(row["sample_id"])

            batch = torch.stack(tensors).to(device)
            batch_features = (
                model(batch)
                .detach()
                .cpu()
                .numpy()
                .astype(np.float32)
            )
            feature_parts.append(batch_features)
            sample_ids.extend(batch_ids)
            print(
                f"STARE ViT: "
                f"{min(batch_start + len(batch_frame), len(frame))}"
                f"/{len(frame)}"
            )

    features = np.vstack(feature_parts).astype(np.float32)
    if features.shape != (len(frame), 768):
        raise ValueError(features.shape)

    pretrained_cfg = {}
    try:
        pretrained_cfg = dict(model.pretrained_cfg)
    except Exception:
        pass

    return (
        features,
        sample_ids,
        time.perf_counter() - start,
        pretrained_cfg,
    )


output_root = CONFIG["OUTPUT_ROOT"]
output_root.mkdir(parents=True, exist_ok=True)

production = load_production_manifest()
X_train_hc, X_train_vit = load_training_features(production)

equivalence = verify_exact_handcrafted_cache(
    production,
    X_train_hc,
    output_root,
)
print("Exact handcrafted equivalence passed.")

stare = build_stare_manifest()
save_csv(
    stare.drop(columns=["resolved_image_path"]),
    output_root / "stare_inclusion_manifest.csv",
)

feature_root = output_root / "feature_cache"
feature_root.mkdir(exist_ok=True)
stare_hc_path = feature_root / "stare_handcrafted_32d.npy"
stare_vit_path = feature_root / "stare_vit_b16_768d.npy"
stare_ids_path = feature_root / "stare_sample_ids.csv"
timing_path = feature_root / "feature_extraction_timing.json"

if (
    CONFIG["RESUME"]
    and stare_hc_path.exists()
    and stare_vit_path.exists()
    and stare_ids_path.exists()
    and timing_path.exists()
):
    X_stare_hc = np.load(stare_hc_path).astype(np.float32)
    X_stare_vit = np.load(stare_vit_path).astype(np.float32)
    cached_ids = pd.read_csv(
        stare_ids_path, dtype={"sample_id": str}
    )["sample_id"].tolist()
    if cached_ids != stare["sample_id"].tolist():
        raise ValueError("Cached STARE IDs do not match")
    timing = json.loads(timing_path.read_text(encoding="utf-8"))
else:
    start = time.perf_counter()
    X_stare_hc = np.vstack([
        extract_handcrafted_32d(path)
        for path in stare["resolved_image_path"]
    ]).astype(np.float32)
    handcrafted_seconds = time.perf_counter() - start

    (
        X_stare_vit,
        vit_ids,
        vit_seconds,
        pretrained_cfg,
    ) = extract_vit_features(stare)
    if vit_ids != stare["sample_id"].tolist():
        raise ValueError("STARE ViT ID order mismatch")

    np.save(stare_hc_path, X_stare_hc)
    np.save(stare_vit_path, X_stare_vit)
    save_csv(stare[["sample_id"]], stare_ids_path)

    timing = {
        "n_stare": len(stare),
        "handcrafted_seconds": handcrafted_seconds,
        "vit_seconds": vit_seconds,
        "hybrid_seconds": handcrafted_seconds + vit_seconds,
        "handcrafted_ms_per_image": (
            1000 * handcrafted_seconds / len(stare)
        ),
        "vit_ms_per_image": (
            1000 * vit_seconds / len(stare)
        ),
        "hybrid_ms_per_image": (
            1000
            * (handcrafted_seconds + vit_seconds)
            / len(stare)
        ),
        "vit_model_requested": CONFIG["VIT_MODEL"],
        "vit_pretrained_cfg": pretrained_cfg,
    }
    save_json(timing, timing_path)

if X_stare_hc.shape != (len(stare), 32):
    raise ValueError(X_stare_hc.shape)
if X_stare_vit.shape != (len(stare), 768):
    raise ValueError(X_stare_vit.shape)

# Exact production fusion order: handcrafted first, then ViT.
X_train_hybrid = np.concatenate(
    [X_train_hc, X_train_vit], axis=1
).astype(np.float32)
X_stare_hybrid = np.concatenate(
    [X_stare_hc, X_stare_vit], axis=1
).astype(np.float32)

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(
    production["class_label"].to_numpy()
)
classes = label_encoder.classes_.tolist()
y_stare = label_encoder.transform(
    stare["harmonized_label"].to_numpy()
)

prediction_frames = []
run_rows = []
model_root = output_root / "run_artifacts" / "models"
model_root.mkdir(parents=True, exist_ok=True)

for method in METHODS:
    for seed in CONFIG["ROBUSTNESS_SEEDS"]:
        seed_all(seed)

        if method == "FROZEN_VIT_SVM":
            X_training = X_train_vit
            X_evaluation = X_stare_vit
            use_pca = False
        else:
            X_training = X_train_hybrid
            X_evaluation = X_stare_hybrid
            use_pca = True

        start = time.perf_counter()
        smote = SMOTE(
            random_state=seed,
            k_neighbors=CONFIG["SMOTE_K_NEIGHBORS"],
        )
        X_balanced, y_balanced = smote.fit_resample(
            X_training,
            y_train,
        )

        scaler = StandardScaler()
        X_balanced_scaled = scaler.fit_transform(X_balanced)
        X_evaluation_scaled = scaler.transform(X_evaluation)

        pca = None
        pca_explained = None
        if use_pca:
            pca = PCA(
                n_components=CONFIG["PCA_COMPONENTS"],
                random_state=seed,
            )
            X_model = pca.fit_transform(X_balanced_scaled)
            X_eval_model = pca.transform(X_evaluation_scaled)
            pca_explained = float(
                pca.explained_variance_ratio_.sum()
            )
        else:
            X_model = X_balanced_scaled
            X_eval_model = X_evaluation_scaled

        classifier = SVC(
            kernel="rbf",
            C=CONFIG["SVM_C"],
            gamma=CONFIG["SVM_GAMMA"],
            class_weight=CONFIG["CLASS_WEIGHT"],
            probability=True,
            random_state=seed,
        )
        classifier.fit(X_model, y_balanced)
        fit_seconds = time.perf_counter() - start

        prediction_start = time.perf_counter()
        predicted_encoded = classifier.predict(X_eval_model)
        probabilities = classifier.predict_proba(X_eval_model)
        prediction_seconds = time.perf_counter() - prediction_start
        predicted_labels = label_encoder.inverse_transform(
            predicted_encoded
        )

        unique_predictions = np.unique(predicted_labels)
        if (
            CONFIG["FAIL_ON_SINGLE_CLASS_PREDICTION"]
            and len(unique_predictions) < 2
        ):
            save_json(
                {
                    "method": method,
                    "seed": seed,
                    "unique_predictions": unique_predictions.tolist(),
                    "counts": (
                        pd.Series(predicted_labels)
                        .value_counts()
                        .to_dict()
                    ),
                },
                output_root
                / f"BLOCKING_SINGLE_CLASS_COLLAPSE_{method}_seed_{seed}.json",
            )
            raise RuntimeError(
                f"{method}, seed={seed} predicted one class only: "
                f"{unique_predictions.tolist()}"
            )

        run_metrics = compute_metrics(
            stare["harmonized_label"],
            predicted_labels,
            classes,
        )
        run_rows.append({
            "method": method,
            "seed": seed,
            "is_primary_seed": seed == CONFIG["PRIMARY_SEED"],
            "n_train": 594,
            "n_stare": len(stare),
            **run_metrics,
            "fit_seconds": fit_seconds,
            "prediction_seconds": prediction_seconds,
            "prediction_ms_per_image_after_features": (
                1000 * prediction_seconds / len(stare)
            ),
            "pca_explained_variance": pca_explained,
            "support_vectors": int(classifier.n_support_.sum()),
            "C": CONFIG["SVM_C"],
            "gamma": CONFIG["SVM_GAMMA"],
            "class_weight": CONFIG["CLASS_WEIGHT"],
            "smote_k_neighbors": CONFIG["SMOTE_K_NEIGHBORS"],
            "feature_order": (
                "Color9_LBP18_Haralick5_then_ViT768"
            ),
        })

        frame = stare[
            [
                "sample_id",
                "image_path",
                "harmonized_label",
                "original_folder_label",
                "label_rule",
                "file_sha256",
            ]
        ].copy()
        frame["method"] = method
        frame["seed"] = seed
        frame["is_primary_seed"] = (
            seed == CONFIG["PRIMARY_SEED"]
        )
        frame["predicted_label"] = predicted_labels
        frame["correct"] = (
            frame["harmonized_label"]
            == frame["predicted_label"]
        ).astype(int)

        for class_index, class_name in enumerate(
            label_encoder.inverse_transform(classifier.classes_)
        ):
            frame[f"prob_{class_name}"] = probabilities[:, class_index]

        prediction_frames.append(frame)

        with (
            model_root / f"{method}__seed_{seed}.pkl"
        ).open("wb") as handle:
            pickle.dump(
                {
                    "smote": smote,
                    "scaler": scaler,
                    "pca": pca,
                    "classifier": classifier,
                    "label_encoder": label_encoder,
                },
                handle,
                protocol=pickle.HIGHEST_PROTOCOL,
            )

        print(
            f"{method} | seed={seed} | "
            f"accuracy={run_metrics['accuracy']:.4f} | "
            f"macro-F1={run_metrics['macro_f1']:.4f}"
        )

predictions = pd.concat(prediction_frames, ignore_index=True)
runs = pd.DataFrame(run_rows)
save_csv(
    predictions,
    output_root / "stare_predictions_all_runs.csv",
)
save_csv(
    runs,
    output_root / "stare_run_metrics.csv",
)

summary = (
    runs.groupby("method", as_index=False)
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_sd=("accuracy", "std"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_sd=("macro_f1", "std"),
        balanced_accuracy_mean=(
            "balanced_accuracy", "mean"
        ),
        balanced_accuracy_sd=(
            "balanced_accuracy", "std"
        ),
        fit_seconds_mean=("fit_seconds", "mean"),
        fit_seconds_sd=("fit_seconds", "std"),
        prediction_ms_mean=(
            "prediction_ms_per_image_after_features",
            "mean",
        ),
        prediction_ms_sd=(
            "prediction_ms_per_image_after_features",
            "std",
        ),
        pca_explained_variance_mean=(
            "pca_explained_variance", "mean"
        ),
    )
)
save_csv(
    summary,
    output_root / "table_1_stare_five_seed_summary.csv",
)

primary = runs[
    runs["seed"] == CONFIG["PRIMARY_SEED"]
].copy()
save_csv(
    primary,
    output_root / "table_2_primary_seed42_metrics.csv",
)

probability_columns = [
    column
    for column in predictions.columns
    if column.startswith("prob_")
]
aggregation = {
    "predicted_label": majority_vote,
    "correct": "sum",
}
for column in probability_columns:
    aggregation[column] = "mean"

consensus = (
    predictions.groupby(
        [
            "sample_id",
            "image_path",
            "harmonized_label",
            "method",
        ],
        as_index=False,
        sort=False,
    )
    .agg(aggregation)
    .rename(
        columns={
            "predicted_label": "consensus_predicted_label",
            "correct": "correct_seed_count",
        }
    )
)
consensus["consensus_correct"] = (
    consensus["harmonized_label"]
    == consensus["consensus_predicted_label"]
).astype(int)
consensus["correct_seed_rate"] = (
    consensus["correct_seed_count"]
    / len(CONFIG["ROBUSTNESS_SEEDS"])
)
save_csv(
    consensus,
    output_root / "stare_consensus_predictions.csv",
)

consensus_rows = []
class_rows = []
confusion_root = output_root / "confusion_matrices"
confusion_root.mkdir(exist_ok=True)

for method in METHODS:
    group = consensus[consensus["method"] == method]
    y_true_labels = group["harmonized_label"]
    y_pred_labels = group["consensus_predicted_label"]

    consensus_rows.append({
        "method": method,
        "n": len(group),
        **compute_metrics(
            y_true_labels,
            y_pred_labels,
            classes,
        ),
    })

    precision_values = precision_score(
        y_true_labels,
        y_pred_labels,
        labels=classes,
        average=None,
        zero_division=0,
    )
    recall_values = recall_score(
        y_true_labels,
        y_pred_labels,
        labels=classes,
        average=None,
        zero_division=0,
    )
    f1_values = f1_score(
        y_true_labels,
        y_pred_labels,
        labels=classes,
        average=None,
        zero_division=0,
    )
    for label, precision_value, recall_value, f1_value in zip(
        classes,
        precision_values,
        recall_values,
        f1_values,
    ):
        class_rows.append({
            "method": method,
            "class": label,
            "support": int((y_true_labels == label).sum()),
            "precision": float(precision_value),
            "recall": float(recall_value),
            "f1": float(f1_value),
        })

    matrix = confusion_matrix(
        y_true_labels,
        y_pred_labels,
        labels=classes,
    )
    pd.DataFrame(
        matrix,
        index=[f"true_{label}" for label in classes],
        columns=[f"pred_{label}" for label in classes],
    ).to_csv(
        confusion_root
        / f"{method}_consensus_confusion_matrix.csv"
    )

consensus_metrics = pd.DataFrame(consensus_rows)
class_metrics = pd.DataFrame(class_rows)
save_csv(
    consensus_metrics,
    output_root / "table_3_consensus_metrics.csv",
)
save_csv(
    class_metrics,
    output_root / "table_4_class_metrics.csv",
)

frozen = consensus[
    consensus["method"] == "FROZEN_VIT_SVM"
][
    [
        "sample_id",
        "harmonized_label",
        "consensus_predicted_label",
        "consensus_correct",
    ]
].rename(
    columns={
        "consensus_predicted_label": "frozen_prediction",
        "consensus_correct": "frozen_correct",
    }
)
hybrid = consensus[
    consensus["method"]
    == "PROPOSED_HYBRID_PCA128_SVM"
][
    [
        "sample_id",
        "harmonized_label",
        "consensus_predicted_label",
        "consensus_correct",
    ]
].rename(
    columns={
        "consensus_predicted_label": "hybrid_prediction",
        "consensus_correct": "hybrid_correct",
    }
)
paired = frozen.merge(
    hybrid,
    on=["sample_id", "harmonized_label"],
    validate="one_to_one",
)
paired["transition"] = np.select(
    [
        (
            (paired["frozen_correct"] == 0)
            & (paired["hybrid_correct"] == 1)
        ),
        (
            (paired["frozen_correct"] == 1)
            & (paired["hybrid_correct"] == 0)
        ),
        (
            (paired["frozen_correct"] == 1)
            & (paired["hybrid_correct"] == 1)
        ),
    ],
    [
        "corrected_by_hybrid",
        "degraded_by_hybrid",
        "both_correct",
    ],
    default="both_incorrect",
)
save_csv(
    paired,
    output_root / "stare_paired_image_transitions.csv",
)

corrected = int(
    (paired["transition"] == "corrected_by_hybrid").sum()
)
degraded = int(
    (paired["transition"] == "degraded_by_hybrid").sum()
)
discordant = corrected + degraded
mcnemar_p = (
    float(
        binomtest(
            corrected,
            discordant,
            0.5,
            alternative="two-sided",
        ).pvalue
    )
    if discordant
    else 1.0
)
bootstrap = bootstrap_paired_difference(
    paired["harmonized_label"].to_numpy(),
    paired["frozen_prediction"].to_numpy(),
    paired["hybrid_prediction"].to_numpy(),
    CONFIG["BOOTSTRAP_RESAMPLES"],
    CONFIG["BOOTSTRAP_SEED"],
)
paired_statistics = {
    "n": len(paired),
    "corrected_by_hybrid": corrected,
    "degraded_by_hybrid": degraded,
    "net_hybrid_minus_frozen": corrected - degraded,
    "exact_mcnemar_p": mcnemar_p,
    **bootstrap,
}
save_json(
    paired_statistics,
    output_root / "table_5_paired_statistics.json",
)

# Validation.
probability_matrix = predictions[
    probability_columns
].to_numpy(float)
validation = {
    "n_train": 594,
    "n_stare": len(stare),
    "methods": METHODS,
    "seeds": CONFIG["ROBUSTNESS_SEEDS"],
    "primary_seed": CONFIG["PRIMARY_SEED"],
    "expected_prediction_rows": (
        len(stare)
        * len(METHODS)
        * len(CONFIG["ROBUSTNESS_SEEDS"])
    ),
    "observed_prediction_rows": len(predictions),
    "duplicate_prediction_keys": int(
        predictions.duplicated(
            ["sample_id", "method", "seed"]
        ).sum()
    ),
    "probabilities_finite": bool(
        np.isfinite(probability_matrix).all()
    ),
    "probabilities_sum_to_one": bool(
        np.allclose(
            probability_matrix.sum(axis=1),
            1.0,
            atol=1e-6,
        )
    ),
    "handcrafted_equivalence_passed": bool(
        equivalence["allclose"]
    ),
    "feature_order": (
        "Color9_LBP18_Haralick5_then_ViT768"
    ),
    "vit_model": CONFIG["VIT_MODEL"],
    "stare_used_for_training_or_selection": False,
}
validation["complete_and_valid"] = bool(
    validation["observed_prediction_rows"]
    == validation["expected_prediction_rows"]
    and validation["duplicate_prediction_keys"] == 0
    and validation["probabilities_finite"]
    and validation["probabilities_sum_to_one"]
    and validation["handcrafted_equivalence_passed"]
    and not validation["stare_used_for_training_or_selection"]
)
save_json(
    validation,
    output_root / "VALIDATION_REPORT.json",
)
if not validation["complete_and_valid"]:
    raise RuntimeError("STARE v7 validation failed")

# Figure.
figure_root = output_root / "figures"
figure_root.mkdir(exist_ok=True)
ordered = summary.set_index("method").loc[METHODS].reset_index()
x = np.arange(len(METHODS))
width = 0.25

fig, axis = plt.subplots(figsize=(9.0, 5.8))
axis.bar(
    x - width,
    100 * ordered["accuracy_mean"],
    width,
    yerr=100 * ordered["accuracy_sd"],
    capsize=4,
    label="Accuracy",
)
axis.bar(
    x,
    100 * ordered["macro_f1_mean"],
    width,
    yerr=100 * ordered["macro_f1_sd"],
    capsize=4,
    label="Macro-F1",
)
axis.bar(
    x + width,
    100 * ordered["balanced_accuracy_mean"],
    width,
    yerr=100 * ordered["balanced_accuracy_sd"],
    capsize=4,
    label="Balanced accuracy",
)
axis.set_xticks(
    x,
    ["Frozen ViT–SVM", "Hybrid PCA-128–SVM"],
)
axis.set_ylabel("Performance (%)")
axis.set_title(
    "Exploratory STARE dataset-shift evaluation"
)
axis.legend(frameon=False)
fig.tight_layout()
for extension in ["png", "tiff"]:
    fig.savefig(
        figure_root / f"figure_1_stare_performance.{extension}",
        dpi=600,
        bbox_inches="tight",
    )
plt.close(fig)

# Manuscript-ready text.
summary_map = summary.set_index("method")
frozen_summary = summary_map.loc["FROZEN_VIT_SVM"]
hybrid_summary = summary_map.loc[
    "PROPOSED_HYBRID_PCA128_SVM"
]
primary_map = primary.set_index("method")
frozen_primary = primary_map.loc["FROZEN_VIT_SVM"]
hybrid_primary = primary_map.loc[
    "PROPOSED_HYBRID_PCA128_SVM"
]

manuscript_text = f"""### Exploratory STARE dataset-shift evaluation

The STARE assessment was repeated after freezing the corrected 594-image development dataset. The exact handcrafted extraction function was recovered from the original experimental and production-reproduction notebooks. The resulting 32-dimensional vector comprised nine RGB color moments, an 18-bin uniform LBP histogram (P=16, R=2), and the first five directional-mean Mahotas Haralick features at distance 1. Before STARE evaluation, this function reproduced the canonical 594-image handcrafted cache with a maximum absolute difference of {equivalence['maximum_absolute_difference']:.8g}. The frozen ViT branch used the original `vit_base_patch16_224` model identifier and 0.5 channel-wise normalization. STARE was not used for training or model selection.

Using the prespecified production seed of 42, Frozen ViT–SVM achieved {100*frozen_primary['accuracy']:.2f}% accuracy and {100*frozen_primary['macro_f1']:.2f}% macro-F1, whereas the Hybrid PCA-128–SVM achieved {100*hybrid_primary['accuracy']:.2f}% accuracy and {100*hybrid_primary['macro_f1']:.2f}% macro-F1. Across five robustness seeds, Frozen ViT–SVM achieved {100*frozen_summary['accuracy_mean']:.2f}% ± {100*frozen_summary['accuracy_sd']:.2f}% accuracy and {100*frozen_summary['macro_f1_mean']:.2f}% ± {100*frozen_summary['macro_f1_sd']:.2f}% macro-F1. The Hybrid achieved {100*hybrid_summary['accuracy_mean']:.2f}% ± {100*hybrid_summary['accuracy_sd']:.2f}% accuracy and {100*hybrid_summary['macro_f1_mean']:.2f}% ± {100*hybrid_summary['macro_f1_sd']:.2f}% macro-F1.

At the five-seed consensus level, the Hybrid corrected {corrected} Frozen errors and degraded {degraded} Frozen-correct cases. The exact McNemar p-value was {mcnemar_p:.4f}, and the paired bootstrap 95% confidence interval for the accuracy difference was {100*bootstrap['accuracy_ci95_low']:.2f} to {100*bootstrap['accuracy_ci95_high']:.2f} percentage points.

The experiment is interpreted as an exploratory rule-harmonized dataset-shift evaluation, not clinical external validation.
"""
(
    output_root / "MANUSCRIPT_READY_STARE_RESULTS_EN.txt"
).write_text(manuscript_text, encoding="utf-8")

metadata = {
    "analysis": "AMD exploratory STARE rerun v8 self-contained exact source",
    "training_dataset": "AMD_DATASET_594_PRODUCTION_v1",
    "handcrafted_source": (
        "Recovered from original experiment and "
        "production-reproduction notebooks"
    ),
    "feature_names": FEATURE_NAMES,
    "feature_order": [
        "RGB color moments 9D",
        "uniform LBP 18D",
        "Mahotas Haralick first five 5D",
        "ViT-B/16 embedding 768D",
    ],
    "vit_model": CONFIG["VIT_MODEL"],
    "pipeline": (
        "SMOTE -> StandardScaler -> PCA128 -> RBF-SVM"
    ),
    "primary_seed": CONFIG["PRIMARY_SEED"],
    "robustness_seeds": CONFIG["ROBUSTNESS_SEEDS"],
    "software": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "torch": torch.__version__,
        "timm": timm.__version__,
    },
}
save_json(
    metadata,
    output_root / "ANALYSIS_METADATA.json",
)

complete_package = (
    output_root.parent / f"{output_root.name}_COMPLETE.zip"
)
if complete_package.exists():
    complete_package.unlink()
with zipfile.ZipFile(
    complete_package,
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    for path in output_root.rglob("*"):
        if path.is_file():
            archive.write(
                path,
                arcname=path.relative_to(
                    output_root.parent
                ),
            )

print("\nSTARE v8 self-contained exact-source rerun completed.")
print("Validation:", validation["complete_and_valid"])
print("Output:", output_root)
print("Package:", complete_package)
